In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path

In [2]:
DATA_PATH = Path("../data")

# EDA

In [3]:
files = [
    "CGCS-Template.csv",
    "Q1-Graph1.csv",
    "Q1-Graph2.csv",
    "Q1-Graph3.csv",
    "Q1-Graph4.csv",
    "Q1-Graph5.csv",
    "Q2-Seed1.csv",
    "Q2-Seed2.csv",
    "Q2-Seed3.csv",
    "CGCS-GraphData-NodeTypes.csv",
    "CGCS-Template-NodeTypes.csv",
    "NodeTypeDescriptions.csv",
    "DemographicCategories.csv"
]

De los archivos presentes en el dataset, existen tanto los Core Graph Files como los Metadata Files. Para el análisis exploratorio de datos, se han utilizado ambos tipos de archivos, ya que los Metadata Files proporcionan información adicional sobre los nodos y las relaciones presentes en los Core Graph Files.

In [ ]:
loaded = {}

for f in files:
  path = DATA_PATH / f
  if path.exists():
    try:
      df = pd.read_csv(path)
      loaded[f] = df
      print(f"{f}: {df.shape}")
    except Exception as e:
      print(f"{f}: ERROR -> {e}")
  else:
    print(f"{f}: NOT FOUND")

CGCS-Template.csv: (1325, 11)
Q1-Graph1.csv: (1216, 11)
Q1-Graph2.csv: (1300, 11)
Q1-Graph3.csv: (729, 11)
Q1-Graph4.csv: (732, 11)
Q1-Graph5.csv: (395, 11)
Q2-Seed1.csv: (1, 11)
Q2-Seed2.csv: (1, 11)
Q2-Seed3.csv: (1, 11)
CGCS-GraphData-NodeTypes.csv: (200912, 2)
CGCS-Template-NodeTypes.csv: (88, 2)
NodeTypeDescriptions.csv: (5, 3)
DemographicCategories.csv: (29, 2)


In [ ]:
for name, df in loaded.items():
  print("="*60)
  print(name)
  print(df.head())
  print(df.columns.tolist())

CGCS-Template.csv
   Source  eType  Target    Time  Weight  SourceLocation  TargetLocation  \
0       0      4     -99     -99     -99             NaN             NaN   
1      41      0      34   86400       1             NaN             NaN   
2      37      0      27   94461       1             NaN             NaN   
3      34      1      27  107548       1             5.0             5.0   
4      41      0      37  127838       1             NaN             NaN   

   SourceLatitude  SourceLongitude  TargetLatitude  TargetLongitude  
0             NaN              NaN             NaN              NaN  
1             NaN              NaN             NaN              NaN  
2             NaN              NaN             NaN              NaN  
3             NaN              NaN             NaN              NaN  
4             NaN              NaN             NaN              NaN  
['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'So

Los tipos de columnas en los grafos presentes en el dataset son los siguientes:
['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']

Estos tipos de columnas se encuentran en los archivos que contienen Q como sufijo, es decir, en los Core Graph Files. Estas columnas representan información sobre las relaciones entre nodos, incluyendo el nodo de origen (Source), el tipo de relación (eType), el nodo de destino (Target), la marca de tiempo (Time), el peso de la relación (Weight), la ubicación del nodo de origen (SourceLocation), la ubicación del nodo de destino (TargetLocation), y las coordenadas geográficas tanto del nodo de origen como del nodo de destino (SourceLatitude, SourceLongitude, TargetLatitude, TargetLongitude).

Lo demás archivos, es decir, los Metadata Files, contienen información adicional sobre los nodos y las relaciones, pero no siguen el mismo formato que los Core Graph Files. Estos archivos pueden incluir columnas como 'id', 'type', 'name', 'description', entre otras, dependiendo del tipo de nodo o relación que se esté describiendo.

In [6]:
summary = []

for name, df in loaded.items():
    summary.append({
        "file": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "columns": ", ".join(df.columns[:6])
    })

summary_df = pd.DataFrame(summary)
summary_df.sort_values("rows", ascending=False)

,file,rows,cols,columns
9,CGCS-GraphData-NodeTypes.csv,200912,2,"NodeID, NodeType"
0,CGCS-Template.csv,1325,11,"Source, eType, Target, Time, Weight, SourceLoc..."
2,Q1-Graph2.csv,1300,11,"Source, eType, Target, Time, Weight, SourceLoc..."
1,Q1-Graph1.csv,1216,11,"Source, eType, Target, Time, Weight, SourceLoc..."
4,Q1-Graph4.csv,732,11,"Source, eType, Target, Time, Weight, SourceLoc..."
3,Q1-Graph3.csv,729,11,"Source, eType, Target, Time, Weight, SourceLoc..."
5,Q1-Graph5.csv,395,11,"Source, eType, Target, Time, Weight, SourceLoc..."
10,CGCS-Template-NodeTypes.csv,88,2,"NodeID, NodeType"
12,DemographicCategories.csv,29,2,"NodeID, Category"
11,NodeTypeDescriptions.csv,5,3,"NodeType, Description, Used in"


Los archivos Q1 corresponden a subgrafos candidatos de tamaño moderado, mientras que el template contiene un patrón pequeño de referencia. Los archivos NodeTypes proveen tipología de nodos necesaria para análisis estructural.

In [7]:
graph_files = [k for k in loaded if "Graph" in k or "Template" in k or "Seed" in k]
meta_files = [k for k in loaded if k not in graph_files]

print("Graph files:")
print(graph_files)

print("\nMetadata files:")
print(meta_files)

Graph files:
['CGCS-Template.csv', 'Q1-Graph1.csv', 'Q1-Graph2.csv', 'Q1-Graph3.csv', 'Q1-Graph4.csv', 'Q1-Graph5.csv', 'Q2-Seed1.csv', 'Q2-Seed2.csv', 'Q2-Seed3.csv', 'CGCS-GraphData-NodeTypes.csv', 'CGCS-Template-NodeTypes.csv']

Metadata files:
['NodeTypeDescriptions.csv', 'DemographicCategories.csv']


Metadata Files:
- NodeTypesDescriptions: Descripciones de tipos de nodos.
- DemographicCategories: Categorías demográficas (e.g., edad, género).

Core Graph Files:
Todos los demás

In [ ]:
expected_core = {"Source", "Target", "eType", "Time"}

for name, df in loaded.items():
  if any(x in name for x in ["Graph", "Template", "Seed"]):
    cols = set(df.columns)
    missing = expected_core - cols
    
    print("="*60)
    print(name)
    print("Columns:", list(df.columns))
    print("Missing required:", missing if missing else "None")